# Cvičení 2: Obrázky jako tenzory, matice a GPU

Na minulém cvičení jsme načetli obrázek a vykreslili ho. Dnes se podíváme, **co ten obrázek ve skutečnosti je**: tabulka čísel, které se v deep learningu říká **tenzor**. Naučíme se z ní vybírat části a upravovat ji. Na konci uvidíme, proč se pro neuronové sítě používají grafické karty.

Většina kódu je připravená. Vaším úkolem je ho spustit, podívat se na výsledek a v úlohách obvykle změnit nebo doplnit jeden řádek.

**Než začnete:** zapněte GPU přes **Runtime → Change runtime type → T4 GPU**. Budeme ho potřebovat až v poslední kapitole, ale přepnutí restartuje prostředí, takže je lepší to udělat hned.

**Jak pracovat s notebookem**

- Buňka se spustí klávesami **Shift + Enter**.
- Buňky spouštějte postupně odshora. Pozdější buňky používají proměnné z dřívějších.
- Pod každou úlohou je prázdná buňka pro váš kód a pod ní skrytá buňka **Řešení**. Dvojklikem na ni zobrazíte kód. Zkuste to nejdřív sami.

**Nejdůležitější rada celého semestru:** když si nejste jistí, co v proměnné je, napište

```python
print(x.shape, x.dtype)
```

`shape` je tvar (rozměry) a `dtype` je datový typ čísel. Většina chyb v deep learningu je chyba v jednom z nich.

## 0. Proč matice a proč GPU

Neuronová síť se skládá z vrstev. Každá vrstva vezme čísla na vstupu, **vynásobí je maticí vah** a výsledek trochu upraví. Když síť zpracovává stovky obrázků najednou, je to násobení velkých matic. Při tréninku sítí tvoří násobení matic naprostou většinu výpočtů.

Grafická karta (GPU) vznikla pro počítačové hry. Tam se každý bod scény násobí maticí a každý pixel obrazovky se počítá zvlášť. GPU proto funguje jinak než procesor (CPU):

- **CPU** má několik málo výkonných jader. Je rychlé na jednu složitou úlohu po krocích.
- **GPU** má tisíce jednoduchých jednotek. Je rychlé, když se má **stejná jednoduchá operace** udělat na obrovském množství čísel najednou.

Násobení matic je přesně taková úloha. Proto se deep learning od roku 2012, kdy síť AlexNet natrénovaná na dvou herních grafických kartách vyhrála soutěž v rozpoznávání obrázků, dělá na GPU.

Na konci cvičení si rozdíl změříme sami.

## 1. Obrázek jako tenzor

Používáme testovací obrázek `astronaut` z knihovny `scikit-image`, který je v Colabu předinstalovaný. Všichni tak mají stejná data a výsledky se dají porovnat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

img = data.astronaut()
print(img.shape, img.dtype)

plt.imshow(img)
plt.axis("off")
plt.show()

Výstup: `(512, 512, 3) uint8`. Obrázek má tedy tři rozměry (osy):

| Osa | Délka | Význam |
|---|---|---|
| 0 | 512 | řádky (výška) |
| 1 | 512 | sloupce (šířka) |
| 2 | 3 | barevné kanály: červená, zelená, modrá |

Každé číslo je celé číslo od 0 do 255 a udává, jak silná je daná barva v daném pixelu. Typ `uint8` znamená „celé číslo bez znaménka, 8 bitů“.

Ke konkrétnímu pixelu se dostaneme indexováním. **Indexuje se od nuly** a v pořadí os: nejdřív řádek, pak sloupec.

In [ ]:
print(img[0, 0])        # levý horní pixel: [154 147 151]
print(img[0, 0, 0])     # jen jeho červená složka: 154

### Úlohy

**1.1** Vypište hodnotu pixelu na řádku 100 a ve sloupci 200.

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 1.1 (dvojklikem zobrazíte řešení)
print(img[100, 200])          # [81 57 17]

**1.2** Jaké je nejmenší a největší číslo v obrázku? (`img.min()`, `img.max()`)

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 1.2 (dvojklikem zobrazíte řešení)
print(img.min(), img.max())   # 0 255

**1.3** Spusťte buňku níže. Kód skončí chybou. Přečtěte si chybovou hlášku a řekněte vlastními slovy, co znamená.

In [ ]:
img[0, 0, 3]

In [ ]:
#@title Řešení 1.3 (dvojklikem zobrazíte řešení)
# IndexError: index 3 is out of bounds for axis 2 with size 3
# Osa 2 (barevné kanály) má jen 3 prvky, tedy indexy 0, 1 a 2.
# Index 3 neexistuje.
try:
    img[0, 0, 3]
except IndexError as e:
    print("IndexError:", e)

## 2. Výřezy (slicing)

Místo jednoho pixelu můžeme vybrat celý úsek. Zápis `a:b` znamená „od `a` do `b`, ale **`b` už ne**“. Samotná dvojtečka `:` znamená „všechno“.

In [ ]:
crop = img[100:200, 50:150]       # řádky 100–199, sloupce 50–149
print(crop.shape)                 # (100, 100, 3)
plt.imshow(crop)
plt.show()

red = img[:, :, 0]                # všechny řádky, všechny sloupce, kanál 0
print(red.shape)                  # (512, 512)
plt.imshow(red, cmap="gray")
plt.show()

Třetí číslo je krok: `::2` vezme každý druhý prvek, `::-1` prochází pozpátku.

In [ ]:
small = img[::2, ::2]             # každý druhý řádek a sloupec
print(small.shape)                # (256, 256, 3)
plt.imshow(img[::-1])             # řádky pozpátku: obrázek vzhůru nohama
plt.show()

### Úlohy

U každé úlohy si nejdřív **zkuste odhadnout tvar výsledku** a pak ho ověřte pomocí `.shape`.

**2.1** Vykreslete zelený kanál a modrý kanál (v odstínech šedi).

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 2.1 (dvojklikem zobrazíte řešení)
plt.imshow(img[:, :, 1], cmap="gray"); plt.title("zelená"); plt.show()
plt.imshow(img[:, :, 2], cmap="gray"); plt.title("modrá"); plt.show()

**2.2** Vykreslete obrázek zrcadlově převrácený zleva doprava. (Nápověda: převrátit je potřeba sloupce, ne řádky.)

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 2.2 (dvojklikem zobrazíte řešení)
plt.imshow(img[:, ::-1]); plt.show()

**2.3** Zmenšete obrázek čtyřikrát v každém směru. Jaký má tvar?

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 2.3 (dvojklikem zobrazíte řešení)
print(img[::4, ::4].shape)        # (128, 128, 3)

**2.4** Vyřízněte levou horní čtvrtinu obrázku.

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 2.4 (dvojklikem zobrazíte řešení)
quarter = img[:256, :256]
print(quarter.shape)              # (256, 256, 3)
plt.imshow(quarter); plt.show()

## 3. Datové typy

Typ `uint8` umí jen čísla 0 až 255. Když výsledek výpočtu přesáhne 255, **přeteče** a začne znovu od nuly, podobně jako tachometr v autě.

In [ ]:
bad = img + 200
print(img[0, 0], bad[0, 0])       # [154 147 151] [ 98  91  95]
plt.imshow(bad)
plt.show()

154 + 200 = 354, ale do `uint8` se vejde jen zbytek po dělení 256, tedy 98. Obrázek proto vypadá divně.

Neuronové sítě pracují s desetinnými čísly (typ `float32`). Obrázek proto vždy převedeme a hodnoty vydělíme 255, aby ležely mezi 0 a 1:

In [ ]:
f = img.astype(np.float32) / 255
print(f.dtype, f.min(), f.max())  # float32 0.0 1.0

### Úlohy

**3.1** Zesvětlete obrázek správně: k `f` přičtěte 0.3 a výsledek ořízněte do rozsahu 0 až 1 funkcí `np.clip(x, 0, 1)`. Vykreslete.

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 3.1 (dvojklikem zobrazíte řešení)
bright = np.clip(f + 0.3, 0, 1)
plt.imshow(bright); plt.show()

**3.2** Udělejte z obrázku negativ: `1 - f`. Vykreslete.

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 3.2 (dvojklikem zobrazíte řešení)
plt.imshow(1 - f); plt.show()

## 4. Broadcasting

V kapitole 3 jste napsali `f + 0.3` a `1 - f`. Sečetli jste tedy jedno číslo s polem 512 × 512 × 3. Matematicky to nedává smysl, a přesto to fungovalo. NumPy si číslo v duchu „rozkopíroval“ na tvar celého obrázku a pak sčítal prvek po prvku. Tomu se říká **broadcasting**.

Stejně to funguje i s větším polem než jedním číslem.

### Jedno číslo pro každý kanál: tónování

Chceme obrázek „zčervenat“: červenou nechat, zelenou a modrou ztlumit. Každý pixel tedy vynásobíme stejnou trojicí čísel:

In [ ]:
tint = np.array([1.0, 0.6, 0.6])   # červená, zelená, modrá
tinted = f * tint
print(f.shape, tint.shape, tinted.shape)   # (512, 512, 3) (3,) (512, 512, 3)
plt.imshow(tinted)
plt.show()

Vektor `tint` má jen tři čísla, ale NumPy ho použije pro každý z 512 × 512 pixelů. Je to přesně to, co dělá barevný filtr na fotce.

### Pravidlo

Tvary obou polí se zarovnají **zprava** a porovnají osu po ose:

```
f       (512, 512, 3)
tint              (3,)
                   ✓     3 a 3 se shodují
```

Dvě osy k sobě pasují, když:

- mají **stejnou délku**, nebo
- jedna z nich má **délku 1**, nebo
- jedna z nich **chybí**.

Osa délky 1 nebo chybějící osa se pak „natáhne“ na délku té druhé. Představte si malé pole jako razítko, které se otiskne na každé místo velkého pole. Otisknout ho ale jde jen tehdy, když sedí jeho tvar.

### Jedno číslo pro každý řádek: přechod do tmy

Teď chceme, aby obrázek směrem dolů postupně tmavl. Potřebujeme tedy jedno číslo pro každý řádek: pro horní řádek 1, pro dolní 0. Spusťte buňku níže:

In [ ]:
fade = np.linspace(1, 0, 512)      # 512 čísel rovnoměrně od 1 do 0
print(fade.shape)                  # (512,)

try:
    f * fade
except ValueError as e:
    print("ValueError:", e)

Proč to nefunguje, je vidět po zarovnání zprava:

```
f       (512, 512, 3)
fade              (512,)
                   ✗     3 a 512 se neshodují
```

NumPy zkouší přiřadit 512 čísel ke třem barvám. Neví, že jsme mysleli řádky.

**Oprava:** řekneme mu to tak, že k `fade` přidáme dvě osy délky 1. K tomu slouží `None` v indexu:

In [ ]:
fade_rows = fade[:, None, None]    # tvar (512, 1, 1)
faded = f * fade_rows
print(fade_rows.shape, faded.shape)   # (512, 1, 1) (512, 512, 3)
plt.imshow(faded)
plt.show()

```
f          (512, 512, 3)
fade_rows  (512,   1, 1)
             ✓     ✓  ✓   osy délky 1 se natáhnou
```

Teď je jasné, že 512 patří k řádkům. Každý řádek obrázku se vynásobí svým číslem a obrázek odshora plynule zčerná.

### Proč je to důležité pro neuronové sítě

V příštím cvičení uvidíte výpočet `X @ W + b`. `X @ W` má tvar `(počet obrázků, počet neuronů)` a bias `b` má tvar `(počet neuronů,)`. Díky broadcastingu se stejný bias přičte ke každému obrázku v dávce a nemusíme ho ručně kopírovat.

### Úlohy

**4.1** Upravte `tint` tak, aby obrázek zmodral. Pak tak, aby zmizela zelená úplně.

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 4.1 (dvojklikem zobrazíte řešení)
plt.imshow(f * np.array([0.6, 0.6, 1.0])); plt.title("zmodralý"); plt.show()
plt.imshow(f * np.array([1.0, 0.0, 1.0])); plt.title("bez zelené"); plt.show()

**4.2** Upravte přechod tak, aby obrázek tmavl **zleva doprava** místo shora dolů. (Nápověda: 512 čísel teď patří ke sloupcům, tedy k ose 1.)

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 4.2 (dvojklikem zobrazíte řešení)
fade_cols = fade[None, :, None]    # tvar (1, 512, 1)
plt.imshow(f * fade_cols); plt.show()

**4.3** Rozhodněte **bez spuštění**, které z těchto operací projdou a jaký bude tvar výsledku. Pak každou vyzkoušejte zvlášť.

```python
f + np.zeros((512, 512, 1))
f + np.zeros((1, 3))
np.zeros((3, 512, 512)) + np.ones(3)
```

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 4.3 (dvojklikem zobrazíte řešení)
print((f + np.zeros((512, 512, 1))).shape)   # (512, 512, 3): osa délky 1 se natáhne na 3
print((f + np.zeros((1, 3))).shape)          # (512, 512, 3): chybějící osa a osa délky 1
try:
    np.zeros((3, 512, 512)) + np.ones(3)
except ValueError as e:
    print("ValueError:", e)
# Zprava se porovnává 512 a 3. Tohle je přesně případ obrázku v PyTorchi,
# kde jsou kanály na začátku: vektor pro kanály by potřeboval tvar (3, 1, 1).

**4.4** (Past) Spusťte buňku níže. Čekali byste tvar `(5,)`? Co se stalo? Proč je to nebezpečné?

In [ ]:
a = np.arange(5)          # tvar (5,)
b = a.reshape(5, 1)       # tvar (5, 1)
print((a + b).shape)
print(a + b)

In [ ]:
#@title Řešení 4.4 (dvojklikem zobrazíte řešení)
# Výsledek má tvar (5, 5).
# Obě pole se natáhla: a po řádcích, b po sloupcích. Vznikla tabulka
# "každý s každým". Kód nespadne, jen tiše spočítá něco jiného.
# U pole s 60 000 prvky by vznikla tabulka s 3.6 miliardami čísel.
# Proto vždy kontrolujte tvary: print(x.shape).

## 5. Násobení matic na obrázku

Každý pixel jsou tři čísla $(r, g, b)$. Černobílý obrázek z nich dostaneme tak, že je sečteme s vahami. Zelená má největší váhu, protože na ni je lidské oko nejcitlivější:

$$\text{jas} = 0.299\,r + 0.587\,g + 0.114\,b$$

Operátor `@` je násobení matic. Tady udělá tento vážený součet pro všech 512 × 512 pixelů najednou:

In [ ]:
w = np.array([0.299, 0.587, 0.114])
gray = f @ w
print(gray.shape)                 # (512, 512)
plt.imshow(gray, cmap="gray")
plt.show()

Ze tvaru `(512, 512, 3) @ (3,)` vznikne `(512, 512)`: poslední osa (3 barvy) se „sečetla“ s vahami a zmizela.

**Přesně tohle dělá jeden neuron:** sečte vstupy s vahami. Rozdíl je jen v tom, že váhy u neuronu nevolíme my, ale síť se je naučí sama.

Když místo vektoru vah použijeme matici 3 × 3, dostaneme z každého pixelu zase tři barvy, jen jinak smíchané:

In [ ]:
A = np.array([[0.393, 0.769, 0.189],    # sépiový filtr
              [0.349, 0.686, 0.168],
              [0.272, 0.534, 0.131]])
sepia = np.clip(f @ A.T, 0, 1)
plt.imshow(sepia)
plt.show()

### Úlohy

**5.1** Jakou hodnotu má levý horní pixel černobílého obrázku `gray`?

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 5.1 (dvojklikem zobrazíte řešení)
print(gray[0, 0])                 # přibližně 0.5865

**5.2** Změňte váhy `w` na `[1, 0, 0]`. Co dostanete? Porovnejte s červeným kanálem z kapitoly 2.

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 5.2 (dvojklikem zobrazíte řešení)
r = f @ np.array([1, 0, 0])
plt.imshow(r, cmap="gray"); plt.show()
# Dostaneme přesně červený kanál (jen v rozsahu 0 až 1 místo 0 až 255).

**5.3** Upravte matici tak, aby prohodila červený a modrý kanál. (Nápověda: začněte od matice, která obrázek nemění, `np.eye(3)`, a prohoďte v ní řádky.)

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 5.3 (dvojklikem zobrazíte řešení)
P = np.array([[0, 0, 1],
              [0, 1, 0],
              [1, 0, 0]])
plt.imshow(f @ P.T); plt.show()
# Červená vlajka zmodrá a modré prvky zčervenají.

**5.4** Spusťte buňku níže. Proč to nefunguje?

In [ ]:
f @ np.ones(4)

In [ ]:
#@title Řešení 5.4 (dvojklikem zobrazíte řešení)
# ValueError: matmul: ... (size 4 is different from 3)
# Každý pixel má 3 čísla, ale vektor vah 4. Rozměry k sobě nesedí.
try:
    f @ np.ones(4)
except ValueError as e:
    print("ValueError:", e)

## 6. Z NumPy do PyTorche

Pro neuronové sítě budeme používat knihovnu **PyTorch**. Její tenzory se chovají skoro stejně jako NumPy pole, ale umí počítat na GPU. PyTorch má dvě vlastní konvence, které je potřeba si pamatovat:

1. Obrázek ukládá jako `(kanály, výška, šířka)`, tedy barvy **na začátku**.
2. Síti se vždy dává **dávka** obrázků tvaru `(počet, kanály, výška, šířka)`, i když je obrázek jen jeden.

Připravený převod:

In [ ]:
import torch

t = torch.from_numpy(img)          # (512, 512, 3)
t = t.permute(2, 0, 1)             # přeskládat osy: (3, 512, 512)
t = t.float() / 255                # desetinná čísla 0 až 1
batch = t.unsqueeze(0)             # přidat osu pro dávku: (1, 3, 512, 512)
print(batch.shape, batch.dtype)

- `permute(2, 0, 1)` říká: „nová osa 0 je stará osa 2, nová osa 1 je stará osa 0, nová osa 2 je stará osa 1“.
- `unsqueeze(0)` přidá na začátek osu délky 1.

### Úlohy

**6.1** Zkopírujte převod z buňky výše a po každém řádku vypište `t.shape`. Sedí tvary s komentáři?

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 6.1 (dvojklikem zobrazíte řešení)
t = torch.from_numpy(img);  print(t.shape)        # torch.Size([512, 512, 3])
t = t.permute(2, 0, 1);     print(t.shape)        # torch.Size([3, 512, 512])
t = t.float() / 255;        print(t.shape)        # torch.Size([3, 512, 512])  float() tvar nemění, jen typ
batch = t.unsqueeze(0);     print(batch.shape)    # torch.Size([1, 3, 512, 512])

**6.2** `plt.imshow` chce obrázek ve tvaru `(výška, šířka, kanály)`. Převeďte tenzor `t` zpět a vykreslete ho. (Nápověda: `t.permute(1, 2, 0)`.)

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 6.2 (dvojklikem zobrazíte řešení)
plt.imshow(t.permute(1, 2, 0)); plt.show()

**6.3** Spojte tři obrázky do jedné dávky: původní, vzhůru nohama a zrcadlový. Použijte `torch.stack([a, b, c])`. Jaký tvar má výsledek? (Pozor: `torch.from_numpy` neumí převzít převrácené pole přímo, je potřeba přidat `.copy()`, například `img[::-1].copy()`.)

In [ ]:
# Váš kód

In [ ]:
#@title Řešení 6.3 (dvojklikem zobrazíte řešení)
a = torch.from_numpy(img.copy())
b = torch.from_numpy(img[::-1].copy())
c = torch.from_numpy(img[:, ::-1].copy())
s = torch.stack([a, b, c])
print(s.shape)                     # torch.Size([3, 512, 512, 3])

## 7. CPU vs. GPU

Zkontrolujte, že máte zapnuté GPU (viz začátek notebooku). Následující buňka má vypsat `True`. Pokud vypíše `False`, přepněte runtime na GPU a spusťte notebook znovu od začátku.

In [ ]:
import time
print(torch.cuda.is_available())

Připravený kód změří, jak dlouho trvá vynásobit dvě náhodné matice velikosti $n \times n$ na CPU a na GPU:

In [ ]:
def measure(n, device):
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)
    A @ B                                   # zahřívací běh, neměříme
    if device == "cuda":
        torch.cuda.synchronize()            # počkat, až GPU dopočítá
    start = time.time()
    for _ in range(10):
        C = A @ B
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - start) / 10

if torch.cuda.is_available():
    for n in [500, 1000, 2000, 4000]:
        t_cpu = measure(n, "cpu")
        t_gpu = measure(n, "cuda")
        print(f"n={n:5d}  CPU {t_cpu*1000:8.2f} ms  GPU {t_gpu*1000:8.2f} ms  "
              f"zrychlení {t_cpu/t_gpu:7.1f}x")
else:
    print("GPU není k dispozici: přepněte Runtime → Change runtime type → T4 GPU.")

Řádek `torch.cuda.synchronize()` je potřeba proto, že GPU pracuje „na pozadí“: Python mu pošle příkaz a hned pokračuje dál. Bez čekání bychom změřili jen dobu odeslání příkazu, ne dobu výpočtu.

### Úlohy

**7.1** Spusťte měření. Jak se mění zrychlení s velikostí matic?

In [ ]:
#@title Řešení 7.1 (dvojklikem zobrazíte řešení)
# Přesné časy se liší podle toho, jaké GPU vám Colab přidělí.
# Zrychlení roste s velikostí matic. U malých matic má GPU málo práce
# a převažuje režie spojená se spuštěním výpočtu. U velkých matic
# zaměstná tisíce jednotek najednou a zrychlení bývá desítkynásobné.

**7.2** Spusťte buňku níže: jedna matice je na CPU, druhá na GPU. Přečtěte si chybu a opravte ji pomocí `.to("cuda")`.

In [ ]:
A = torch.randn(100, 100)
B = torch.randn(100, 100, device="cuda")
A @ B

In [ ]:
#@title Řešení 7.2 (dvojklikem zobrazíte řešení)
# RuntimeError: Expected all tensors to be on the same device, ...
# Všechny tenzory v jedné operaci musí být na stejném zařízení.
# Tuhle chybu uvidíte během semestru ještě mnohokrát.
if torch.cuda.is_available():
    A = torch.randn(100, 100).to("cuda")
    B = torch.randn(100, 100, device="cuda")
    print((A @ B).device)          # cuda:0

## Taháček

```python
x.shape, x.dtype               # první krok při každé chybě

x[i, j]                        # jeden prvek (indexuje se od 0)
x[a:b]                         # úsek a .. b-1
x[:, :, 0]                     # všechno z os 0 a 1, z osy 2 jen index 0
x[::2]                         # každý druhý
x[::-1]                        # pozpátku

x.astype(np.float32) / 255     # obrázek na desetinná čísla 0 až 1
x * np.array([1, 0.6, 0.6])    # broadcasting: jedno číslo pro každý kanál
x * v[:, None, None]           # broadcasting: jedno číslo pro každý řádek
x @ w                          # násobení matic

t.permute(2, 0, 1)             # (H, W, C) -> (C, H, W)
t.unsqueeze(0)                 # přidat osu pro dávku
t.to("cuda")                   # přesun na GPU
```